#### 8. Write a short design note describing how Cyntexa could replace a nightly batch warehouse load with a lakehouse pipeline, calling out specifically where ACID transactions and time travel reduce operational risk compared to the current warehouse.

This is a biggest problem with the Warehouses that a if the data is a lot big then it will recompute the entire data once again to refined the result into the presentable form. In case of an error all the data got messy and every process will stop and got stuck with the messy data that could also need to be cleaned first. 

We can eleminate these risks by implementing a Lakehouse Pipeline where all the data is coming into micro batches that could take less resources to load and transform. When we are dealing with the micro batches we can also add another layer of Increamental Batch system which will stop the hardwares to reprocess the entire data that could be in terbytes, but simple process the incoming data and then just save the data into the database.

How Lakhouse helps in achieving the ACID transactions:
1. Atomicity: In case of a failure the latest version of the database would not be affect partially as the data is written either fully on success or nothing at all at failure.
2. Consistency: What if Cyntexa's management want to see the dashboards from either their system or phone then data must be available correctly and consistent among all the devices and location.
3. Isolation: With the help of Lakhouse Isolation is achieved by default as at the time of any failure all the previous version would not be affected as there will be an isolation among all the versions of the Database.
4. Durability: All the data is stored safely into the cloud so there si very low risk of the data loss as compared to the Data Warehouse, If the poer goes off then all the data will be vanished. But, on the other hand lakehouse is completely on the clouds that helps to keep the data safer than anything because cloud storage uses Distributed-auto-sync system where teh data is sotred at multiple place and so there is always a backup.

How Time Travelling reduces Operational Risks:
 One of the most greatest feature of time travelling is that nothing can be fully modified at a time and you can write simultaneously into the new version and at the time of failure we can just go to the previous versions. It ensures better Integirty than Warehouses or any other traditional file systems. In a highly changing environment it is very important to maintain the correctness of data at any cost.

#### 9. Simulate two concurrent writers appending to the same Delta table (two notebook cells or jobs), then use DESCRIBE HISTORY to explain how the transaction log resolved the write order and what would happen if the writes conflicted.

In [0]:
%sql
select * from dev.demo.sample_products

In [0]:
from delta.tables import DeltaTable
import threading

def insert_operation_1():
    print("Starting INSERT operation_1...")
    new_data = spark.createDataFrame([
        (12, "New Product A", "Category A", 29.99),
        (13, "New Product B", "Category B", 39.99),
        (14, "New Product C", "Category C", 49.99)
    ], ["product_id", "name", "category", "price"])
    
    new_data.write.format("delta").mode("append").saveAsTable("dev.demo.sample_products")
    print("INSERT operation_1 completed!")

def insert_operation_2():
    print("Starting INSERT operation_2...")
    new_data = spark.createDataFrame([
        (14, "New Product A", "Category A", 29.99),
        (15, "New Product B", "Category B", 39.99),
        (16, "New Product C", "Category C", 49.99)
    ], ["product_id", "name", "category", "price"])
    
    new_data.write.format("delta").mode("append").saveAsTable("dev.demo.sample_products")
    print("INSERT operation_2 completed!")

thread1 = threading.Thread(target=insert_operation_1)
thread2 = threading.Thread(target=insert_operation_2)

print("Starting concurrent write operations...")
thread1.start()
thread2.start()
thread1.join()
thread2.join()

print("\n=== Both operations completed successfully! ===")
print("Delta Lake's transaction log resolved the write order automatically.")

In [0]:
%sql
DESCRIBE HISTORY dev.demo.sample_products LIMIT 5


**How Delta Lake's Transaction Log Resolved the Write Order**

When two concurrent writers append to the same Delta table, Delta Lake's transaction log ensures ACID guarantees through optimistic concurrency control:

1. **Sequential Commit Order**: Even though both INSERT operations started simultaneously via threading, Delta Lake serialized the commits. The transaction log assigned sequential version numbers.

2. **Transaction Log Mechanism**: Each write operation:
   - Reads the current table version
   - Writes new Parquet files to storage
   - Attempts to commit by writing a new JSON entry to the `_delta_log/` directory
   - The commit succeeds only if the table version hasn't changed since the read

3. **Conflict-Free Append**: Since both operations were **appends** (`.mode("append")`), they don't conflict:
   - Thread 1 commits first → creates version N with its 3 new rows
   - Thread 2 commits second → creates version N+1 with its 3 new rows
   - Both succeed because appends are **additive** and don't require reading existing data

**What Would Happen If the Writes Conflicted:**

Conflicts occur when operations are **not commutative** (order matters):

- **UPDATE/DELETE + UPDATE/DELETE on overlapping rows**: If both threads updated `product_id = 14`, one would fail with `ConcurrentAppendException` or `ConcurrentDeleteException`
- **MERGE operations**: If both threads ran MERGE UPSERT logic on the same key, the second would detect that files have changed and retry
- **Schema changes**: If one thread altered the schema while another wrote data, Delta would reject the incompatible write

**Resolution Strategy:**
- The losing transaction gets a `ConcurrentModificationException`
- The application must **retry** the operation, re-reading the latest table version
- Delta Lake automatically handles retries for some operations (like streaming writes)

**Key Takeaway**: Appends are commutative (order doesn't matter for correctness), so both succeeded. Non-commutative operations require Delta Lake to reject one transaction and force a retry to maintain consistency.

#### 10. (Data Analyst) Write a one-page comparison memo: list 3 concrete advantages a lakehouse gives analysts over a traditional warehouse, and 1 tradeoff to watch for.

##Advantages for Analysts:

###Fresh Data for Real-Time Analysis: 
Traditional warehouses use batches to load data. This often means analysts work with yesterday's data. Lakehouses support real-time data streaming. You can query data the moment it arrives.

###Direct Access to All Data Types:
Traditional warehouses only hold structured data like neat tables and rows. Lakehouses store structured, semi-structured (JSON, XML), and unstructured data (images, PDFs) in one place. Analysts can query everything together without waiting for data teams to convert it. 

###No More Data Staleness or Discrepancies:
In old systems, data is copied from a lake to a warehouse. This creates different versions of the data. Lakehouses use a single copy of data. Every analyst works out of the exact same data repository.

##One Tradeoff to Watch For: Complex Performance Tuning
The biggest tradeoff of a lakehouse is higher complexity in maintaining fast query performance. A traditional data warehouse automatically organizes, indexes, and optimizes data behind the scenes to make queries run instantly. A lakehouse separates computing power from storage. It relies on open table formats (like Delta Lake, Iceberg, or Hudi). If your engineering team does not actively manage, clean, and compress these files, analyst queries will slow down significantly over time. You lose the speed advantage of a traditional warehouse without proper maintenance.